# BTCPredictor2 — GPU Training (1-Day Horizon)

## Before running
1. Enable GPU: **Runtime → Change runtime type → A100 GPU** (recommended)
2. Run Cell 1 (clones repo + installs packages)
3. Run Cell 2 (verifies the 10 yearly CSVs — downloads any missing ones from GitHub)
4. Run Cells 3, 4, 5 to train TFT, BiLSTM, and Meta

## Training data
The 10 yearly merged CSVs live in `data/yearly_merged/` and are committed to the repo.
Cell 1 clones the repo (including all data files). Cell 2 verifies they are present
and downloads any missing files directly from GitHub — **no Drive upload needed**.
- `{YYYY}_bilstm_merged.csv` — 15-min sampled, m15_* + h1_* features (BiLSTM input)
- `{YYYY}_tft_merged.csv`   — 4h sampled,  h4_* + d1_* features  (TFT input)

**Estimated time on A100 GPU:**
- TFT (30-day / 4h): 2-4 hours
- BiLSTM (24h / 15min): 1-2 hours
- Meta: 5 minutes

In [ ]:
# ============================================================
# CELL 1 - SETUP
# Always deletes and re-clones so you get the latest GitHub push.
# ============================================================

GITHUB_URL = 'https://github.com/chefo919/BTCPredictor2.git'

import shutil, os
if os.path.exists('/content/BTCPredictor2'):
    shutil.rmtree('/content/BTCPredictor2')
    print('Removed old clone.')

!git clone {GITHUB_URL} /content/BTCPredictor2
!cd /content/BTCPredictor2 && git log --oneline -3

import sys
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

!pip install -q xgboost joblib ta scikit-learn-intelex numba

os.makedirs('/content/BTCPredictor2/data/yearly_merged', exist_ok=True)
os.makedirs('/content/BTCPredictor2/data/yearly_1m',     exist_ok=True)
os.makedirs('/content/BTCPredictor2/models/saved',       exist_ok=True)

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print(f'GPU: {gpus[0].name}  |  Mixed precision: ON')
else:
    print('WARNING: No GPU — Runtime > Change runtime type > GPU')

import config
config.BATCH_TFT    = 512
config.BATCH_BILSTM = 1024

from features.engineer import get_feature_groups, BILSTM_FEAT_COLS, TFT_FEAT_COLS
groups = get_feature_groups()

print()
print(f'BiLSTM features: {len(groups["bilstm"])}  (m15/h1 — {config.SEQ_LEN_BILSTM} steps x 15min = 48h context)')
print(f'TFT    features: {len(groups["tft_dynamic"])}  (h4/d1  — {config.SEQ_LEN_TFT} steps x 4h = 30 days context)')
print()
print(f'TFT:    SEQ={config.SEQ_LEN_TFT} | HORIZON={config.HORIZON_TFT}min (1 day) | batch={config.BATCH_TFT}')
print(f'BiLSTM: SEQ={config.SEQ_LEN_BILSTM} | HORIZON={config.HORIZON_BILSTM}min (1 day) | batch={config.BATCH_BILSTM}')
print()
print('Done. Run Cell 2 to verify training data.')

In [ ]:
# ============================================================
# CELL 2 - VERIFY DATA FILES (from GitHub)
# Checks both yearly_merged (features) and yearly_1m (1m OHLC for labels).
# All files are committed to the repo — no Drive upload needed.
# ============================================================

import os, subprocess

GITHUB_RAW  = 'https://raw.githubusercontent.com/chefo919/BTCPredictor2/main'
YEARLY_DIR  = '/content/BTCPredictor2/data/yearly_merged'
YEARLY_1M   = '/content/BTCPredictor2/data/yearly_1m'
os.makedirs(YEARLY_DIR, exist_ok=True)
os.makedirs(YEARLY_1M,  exist_ok=True)

YEARS = ['2022', '2023', '2024', '2025', '2026']

MERGED_FILES = (
    [f'{y}_bilstm_merged.csv' for y in YEARS] +
    [f'{y}_tft_merged.csv'    for y in YEARS]
)
M1_FILES = [f'{y}_btc_1m.csv' for y in YEARS]


def _download_if_missing(files, local_dir, remote_subdir):
    missing = [f for f in files if not os.path.exists(os.path.join(local_dir, f))]
    if missing:
        print(f'Downloading {len(missing)} missing file(s) from GitHub ({remote_subdir})...')
        for fname in missing:
            url = f'{GITHUB_RAW}/{remote_subdir}/{fname}'
            dst = os.path.join(local_dir, fname)
            r = subprocess.run(['wget', '-q', '--show-progress', '-O', dst, url])
            if r.returncode != 0 or not os.path.exists(dst) or os.path.getsize(dst) < 1000:
                print(f'  ERROR: failed to download {fname}')
            else:
                print(f'  {fname}  ({os.path.getsize(dst)/1024**2:.1f} MB)')
    else:
        print(f'All {len(files)} {remote_subdir} files already present.')


_download_if_missing(MERGED_FILES, YEARLY_DIR, 'data/yearly_merged')
_download_if_missing(M1_FILES,     YEARLY_1M,  'data/yearly_1m')

# Summary — merged files
print()
bilstm_total = tft_total = 0
errors = []
for fname in sorted(MERGED_FILES):
    path = os.path.join(YEARLY_DIR, fname)
    if os.path.exists(path):
        rc = sum(1 for _ in open(path)) - 1
        sz = os.path.getsize(path) / 1024**2
        tag = 'bilstm' if 'bilstm' in fname else 'tft   '
        print(f'  [{tag}] {fname}: {rc:,} rows  ({sz:.1f} MB)')
        if 'bilstm' in fname:
            bilstm_total += rc
        else:
            tft_total += rc
    else:
        errors.append(fname); print(f'  MISSING: {fname}')

print()
print(f'Total bilstm rows: {bilstm_total:,}   Total tft rows: {tft_total:,}')

# Summary — 1m files
print()
m1_total = 0
for fname in sorted(M1_FILES):
    path = os.path.join(YEARLY_1M, fname)
    if os.path.exists(path):
        rc = sum(1 for _ in open(path)) - 1
        sz = os.path.getsize(path) / 1024**2
        print(f'  [1m    ] {fname}: {rc:,} rows  ({sz:.1f} MB)')
        m1_total += rc
    else:
        errors.append(fname); print(f'  MISSING: {fname}')

print()
print(f'Total 1m rows: {m1_total:,}')
if errors:
    print(f'WARNING: {len(errors)} file(s) missing — check repo has data/ committed')
else:
    print('All files verified. Run Cell 3 to apply labels + train TFT.')

In [ ]:
# ============================================================
# CELL 3 - TRAIN TFT ENSEMBLE  (estimated: 2-4h on A100 for 3 seeds)
# Input: 4h-sampled tft_merged (h4/d1 features, SEQ=180 steps = 30 days)
# Labels: triple-barrier (TP=6%, SL=3%, horizon=1440min) from yearly_1m/
# ============================================================

import os, sys, time, json
import numpy as np
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

import tensorflow as tf
tf.keras.mixed_precision.set_global_policy('mixed_float16')

import config
config.BATCH_TFT = 512

import pandas as pd
from features.engineer import get_feature_groups
from models import tft_model
from training.labels import apply_triple_barrier

groups           = get_feature_groups()
TFT_DYN_FEATURES = groups['tft_dynamic']
TFT_STA_FEATURES = groups['tft_static']

YEARLY_DIR  = 'data/yearly_merged'
PATH_1M     = 'data/yearly_1m'          # directory of yearly 1m splits
START_DATE  = '2022-02-01'
CUTOFF_DATE = config.TRAINING_CUTOFF_DATE
start_ts    = pd.Timestamp(START_DATE,  tz='UTC')
cutoff_ts   = pd.Timestamp(CUTOFF_DATE, tz='UTC')

dfs = []
for fname in sorted(os.listdir(YEARLY_DIR)):
    if not fname.endswith('_tft_merged.csv'):
        continue
    df_y = pd.read_csv(os.path.join(YEARLY_DIR, fname), parse_dates=['time'])
    if df_y['time'].dt.tz is None:
        df_y['time'] = pd.to_datetime(df_y['time'], utc=True)
    df_y = df_y[(df_y['time'] >= start_ts) & (df_y['time'] <= cutoff_ts)]
    if not df_y.empty:
        dfs.append(df_y)

df_tft = pd.concat(dfs, ignore_index=True).sort_values('time').reset_index(drop=True)
print(f'TFT rows loaded: {len(df_tft):,}  ({START_DATE} -> {CUTOFF_DATE})')

print()
df_tft = apply_triple_barrier(
    df_tft, PATH_1M,
    config.TAKE_PROFIT_PCT, config.STOP_LOSS_PCT, config.HORIZON_TFT,
)
print(f'TFT rows after labeling: {len(df_tft):,}  label balance: {df_tft["target"].mean():.3f}')
print()

t0 = time.time()
results = tft_model.train_ensemble(df_tft, TFT_DYN_FEATURES, TFT_STA_FEATURES)
elapsed = time.time() - t0

avg_test = float(np.mean([r['test_acc'] for r in results]))
avg_val  = float(np.mean([r['val_acc']  for r in results]))

acc_path = 'models/saved/model_accuracies.json'
existing = json.load(open(acc_path)) if os.path.exists(acc_path) else {}
existing.update({'tft': avg_test, 'tft_val': avg_val})
with open(acc_path, 'w') as f:
    json.dump(existing, f, indent=2)

print()
print('TFT ENSEMBLE COMPLETE')
for i, r in enumerate(results):
    print(f'  Seed {i}: test={r["test_acc"]:.4f}  val={r["val_acc"]:.4f}')
print(f'  Average:  test={avg_test:.4f}  val={avg_val:.4f}')
print(f'  Time:     {int(elapsed//3600)}h {int((elapsed%3600)//60)}m')
print()
print('Run Cell 4 to train BiLSTM ensemble.')

In [ ]:
# ============================================================
# CELL 4 - TRAIN BILSTM ENSEMBLE  (estimated: 1-2h on A100 for 3 seeds)
# Input: 15min-sampled bilstm_merged (m15/h1 features, SEQ=192 steps = 48h)
# Labels: triple-barrier (TP=6%, SL=3%, horizon=1440min) from yearly_1m/
# ============================================================

import os, sys, time, json
import numpy as np
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

import tensorflow as tf
tf.keras.mixed_precision.set_global_policy('mixed_float16')

import config
config.BATCH_BILSTM = 1024

import pandas as pd
from features.engineer import get_feature_groups
from models import bilstm_model
from training.labels import apply_triple_barrier

groups          = get_feature_groups()
BILSTM_FEATURES = groups['bilstm']

YEARLY_DIR  = 'data/yearly_merged'
PATH_1M     = 'data/yearly_1m'
START_DATE  = '2022-02-01'
CUTOFF_DATE = config.TRAINING_CUTOFF_DATE
start_ts    = pd.Timestamp(START_DATE,  tz='UTC')
cutoff_ts   = pd.Timestamp(CUTOFF_DATE, tz='UTC')

dfs = []
for fname in sorted(os.listdir(YEARLY_DIR)):
    if not fname.endswith('_bilstm_merged.csv'):
        continue
    df_y = pd.read_csv(os.path.join(YEARLY_DIR, fname), parse_dates=['time'])
    if df_y['time'].dt.tz is None:
        df_y['time'] = pd.to_datetime(df_y['time'], utc=True)
    df_y = df_y[(df_y['time'] >= start_ts) & (df_y['time'] <= cutoff_ts)]
    if not df_y.empty:
        dfs.append(df_y)

df_bilstm = pd.concat(dfs, ignore_index=True).sort_values('time').reset_index(drop=True)
print(f'BiLSTM rows loaded: {len(df_bilstm):,}  ({START_DATE} -> {CUTOFF_DATE})')

print()
df_bilstm = apply_triple_barrier(
    df_bilstm, PATH_1M,
    config.TAKE_PROFIT_PCT, config.STOP_LOSS_PCT, config.HORIZON_BILSTM,
)
print(f'BiLSTM rows after labeling: {len(df_bilstm):,}  label balance: {df_bilstm["target"].mean():.3f}')
print()

t0 = time.time()
results = bilstm_model.train_ensemble(df_bilstm, BILSTM_FEATURES)
elapsed = time.time() - t0

avg_test = float(np.mean([r['test_acc'] for r in results]))
avg_val  = float(np.mean([r['val_acc']  for r in results]))

acc_path = 'models/saved/model_accuracies.json'
existing = json.load(open(acc_path)) if os.path.exists(acc_path) else {}
existing.update({'bilstm': avg_test, 'bilstm_val': avg_val})
with open(acc_path, 'w') as f:
    json.dump(existing, f, indent=2)

print()
print('BiLSTM ENSEMBLE COMPLETE')
for i, r in enumerate(results):
    print(f'  Seed {i}: test={r["test_acc"]:.4f}  val={r["val_acc"]:.4f}')
print(f'  Average:  test={avg_test:.4f}  val={avg_val:.4f}')
print(f'  Time:     {int(elapsed//3600)}h {int((elapsed%3600)//60)}m')
print()
print('Run Cell 5 to train Meta and download models.')

In [ ]:
# ============================================================
# CELL 5 - TRAIN META (AGREEMENT-BASED ROUTING) + DOWNLOAD  (~5 minutes)
# ============================================================

import os, sys, time, shutil, json
os.chdir('/content/BTCPredictor2')
sys.path.insert(0, '/content/BTCPredictor2')

import numpy as np
import pandas as pd
import config
from features.engineer import get_feature_groups
from models import tft_model, bilstm_model, meta_model
from training.labels import apply_triple_barrier

groups           = get_feature_groups()
BILSTM_FEATURES  = groups['bilstm']
TFT_DYN_FEATURES = groups['tft_dynamic']
TFT_STA_FEATURES = groups['tft_static']
ALL_GATE_FEATURES = TFT_DYN_FEATURES + BILSTM_FEATURES

YEARLY_DIR  = 'data/yearly_merged'
PATH_1M     = 'data/yearly_1m'
START_DATE  = '2022-02-01'
CUTOFF_DATE = config.TRAINING_CUTOFF_DATE
start_ts    = pd.Timestamp(START_DATE,  tz='UTC')
cutoff_ts   = pd.Timestamp(CUTOFF_DATE, tz='UTC')


def _load_yearly(suffix):
    dfs = []
    for fname in sorted(os.listdir(YEARLY_DIR)):
        if not fname.endswith(suffix):
            continue
        df_y = pd.read_csv(os.path.join(YEARLY_DIR, fname), parse_dates=['time'])
        if df_y['time'].dt.tz is None:
            df_y['time'] = pd.to_datetime(df_y['time'], utc=True)
        df_y = df_y[(df_y['time'] >= start_ts) & (df_y['time'] <= cutoff_ts)]
        if not df_y.empty:
            dfs.append(df_y)
    return pd.concat(dfs, ignore_index=True).sort_values('time').reset_index(drop=True)


# Load + label (mirrors train.py exactly)
print('[0/1] Applying triple-barrier labels...')
df_tft    = _load_yearly('_tft_merged.csv')
df_bilstm = _load_yearly('_bilstm_merged.csv')
df_tft    = apply_triple_barrier(df_tft,    PATH_1M, config.TAKE_PROFIT_PCT, config.STOP_LOSS_PCT, config.HORIZON_TFT)
df_bilstm = apply_triple_barrier(df_bilstm, PATH_1M, config.TAKE_PROFIT_PCT, config.STOP_LOSS_PCT, config.HORIZON_BILSTM)

df_tft    = df_tft.dropna(subset=TFT_DYN_FEATURES).reset_index(drop=True)
df_bilstm = df_bilstm.dropna(subset=BILSTM_FEATURES).reset_index(drop=True)

n_tft    = len(df_tft)
n_bilstm = len(df_bilstm)

PURGE_TFT    = 6    # 24h / 4h    = 6 rows
PURGE_BILSTM = 96   # 24h / 15min = 96 rows

tft_train_end  = int(n_tft    * 0.70)
tft_oof_start  = tft_train_end  + PURGE_TFT
tft_oof_end    = int(n_tft    * 0.90)

bi_train_end   = int(n_bilstm * 0.70)
bi_oof_start   = bi_train_end   + PURGE_BILSTM
bi_oof_end     = int(n_bilstm * 0.90)

df_tft_oof    = df_tft.iloc[tft_oof_start:tft_oof_end].copy()
df_bilstm_oof = df_bilstm.iloc[bi_oof_start:bi_oof_end].copy()

print(f'TFT total: {n_tft:,}  |  OOF: rows {tft_oof_start:,}-{tft_oof_end:,} ({len(df_tft_oof):,} rows)')
print(f'BiLSTM total: {n_bilstm:,}  |  OOF: rows {bi_oof_start:,}-{bi_oof_end:,} ({len(df_bilstm_oof):,} rows)')
print(f'Ensemble ready: TFT={tft_model._ensemble_ready()}  BiLSTM={bilstm_model._ensemble_ready()}')
print()

val_X_dyn = df_tft_oof[TFT_DYN_FEATURES].values.astype('float32')
val_X_sta = np.zeros((len(df_tft_oof), 0), dtype='float32')
val_X_bi  = df_bilstm_oof[BILSTM_FEATURES].values.astype('float32')

acc_path       = 'models/saved/model_accuracies.json'
saved_acc      = json.load(open(acc_path)) if os.path.exists(acc_path) else {}
tft_val_err    = 1.0 - saved_acc.get('tft_val', 0.51)
bilstm_val_err = 1.0 - saved_acc.get('bilstm_val', 0.51)

print('Generating TFT OOF predictions...')
val_tft_probs = tft_model.predict_proba_batch(
    val_X_dyn, val_X_sta, timestamps=df_tft_oof['time'])

print('Generating BiLSTM OOF predictions...')
val_bilstm_probs_full = bilstm_model.predict_proba_batch(val_X_bi)

# Align BiLSTM OOF onto TFT timestamps (mirrors train.py)
df_bi_probs          = df_bilstm_oof[['time']].copy()
df_bi_probs['bilstm_prob'] = val_bilstm_probs_full
df_tft_idx           = df_tft_oof[['time']].copy().sort_values('time')
df_tft_idx['tft_idx'] = np.arange(len(df_tft_oof))

aligned = pd.merge_asof(
    df_tft_idx,
    df_bi_probs.sort_values('time'),
    on='time', direction='nearest',
)
val_bilstm_probs_aligned = aligned['bilstm_prob'].values.astype('float32')
val_tft_probs_aligned    = val_tft_probs[aligned['tft_idx'].values]

bilstm_oof_sorted = df_bilstm_oof.sort_values('time').reset_index(drop=True)
df_combined = pd.merge_asof(
    df_tft_oof.sort_values('time').reset_index(drop=True),
    bilstm_oof_sorted[['time'] + BILSTM_FEATURES],
    on='time', direction='nearest',
)

print('Training agreement-based routing meta-learner...')
meta_results = meta_model.train(
    df_combined,
    val_tft_probs_aligned, val_bilstm_probs_aligned,
    tft_val_err, bilstm_val_err,
    ALL_GATE_FEATURES,
)
w = meta_results.get('weights', {})

print()
print('=' * 55)
print('TRAINING COMPLETE (1-day horizon)')
print(f'  TFT accuracy:    {saved_acc.get("tft", 0):.3f}  (ensemble of {config.N_ENSEMBLE} seeds)')
print(f'  BiLSTM accuracy: {saved_acc.get("bilstm", 0):.3f}  (ensemble of {config.N_ENSEMBLE} seeds)')
print(f'  Meta routing:    {meta_results["train_acc"]:.3f}  (OOF, agreement-based gate)')
print(f'  Gate CV acc:     {meta_results.get("gate_cv_acc", 0):.3f}  '
      f'({meta_results.get("n_disagree", 0):,} disagreement rows)')
print(f'  Static fallback  TFT: {w.get("tft", 0):.3f}  BiLSTM: {w.get("bilstm", 0):.3f}')
print(f'  Data window:     {START_DATE} -> {CUTOFF_DATE}')
print('=' * 55)

# ── Package and download ──────────────────────────────────────────────────
OUT = '/content/models_output'
os.makedirs(OUT, exist_ok=True)

for s in range(config.N_ENSEMBLE):
    for fname in [f'tft_s{s}.keras', f'tft_scaler_s{s}.pkl',
                  f'bilstm_s{s}.keras', f'bilstm_scaler_s{s}.pkl']:
        src = f'models/saved/{fname}'
        if os.path.exists(src):
            shutil.copy(src, f'{OUT}/{fname}')

for fname in ['meta_xgb.pkl', 'model_accuracies.json', 'training_cutoff.txt']:
    src = f'models/saved/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{OUT}/{fname}')

saved_files = os.listdir(OUT)
print(f'\nPackaged {len(saved_files)} files: {sorted(saved_files)}')
shutil.make_archive('/content/btc_models', 'zip', OUT)

from google.colab import files
print()
print('Downloading btc_models.zip to your PC...')
files.download('/content/btc_models.zip')
print()
print('Extract the zip and copy all files into your local models/saved/ folder.')
print('Then run: python papertrading/backtest.py --start 2026-04-15')